# 5-A. 최종 피처 필터링 (Feature Filtering)

이 노트북에서는 도메인 지식, 다중공선성 분석, Spearman 상관계수 등을 기반으로 최종 제외할 피처 목록을 정의하고,
훈련(Train) 및 테스트(Validation) 데이터셋에서 해당 컬럼들을 일괄 제거(필터링)하여 저장합니다.

> **참고:** 날짜 컷오프(2014-04-01 이후 데이터 필터링)는 이전 파이프라인 단계에서 완료되었으므로, 여기서는 별도의 날짜 컷오프 연산을 수행하지 않고 최종 정합성 검증 단계에서만 검사합니다.

In [1]:
import pandas as pd
import duckdb
import os

# ---------------------------------------------
# 1. 경로 및 파일 설정
# ---------------------------------------------
os.makedirs("../data2/05_feature_selection", exist_ok=True)
train_path = "../data2/04_feature_engineering/fs_train.parquet"
val_path  = "../data2/04_feature_engineering/fs_validation.parquet"

TRAIN_OUT = "../data2/05_feature_selection/fs_train_filtered.parquet"
VAL_OUT  = "../data2/05_feature_selection/fs_validation_filtered.parquet"

# ---------------------------------------------
# 2. 노트북 내장 제외 피처 딕셔너리 (Exclusion Config)
# ---------------------------------------------
EXCLUSION_CONFIG = {
    "drop_v1": [
        "error_saturation_score",
        "shock_seek_interaction",
        "log_shock_fly_interaction",
        "shock_fatigue_rate",
        "smart_191_raw",
        "s191_diff",
        "s189_diff",
        "s191_28d_sum",
        "s191_14d_sum",
        "s199_14d_max",
        "s199_diff",
        "s199_14d_burst",
        "s199_14d_sum",
        "s199_28d_sum",
        "s199_28d_max",
        "s191_14d_max",
        "s191_28d_max",
        "s194_diff",
        "s190_diff",
        "s189_28d_max",
        "s189_28d_highfly_burst",
        "s189_28d_sum",
        "s192_14d_burst",
        "temp_error_index",
        "is_warmup_7d",
        "total_reads_14d_sum",
        "total_reads_14d_mean"
    ],
    "총 탐색량": [
        "total_seeks_7d_accel",
        "total_seeks_28d_accel",
        "total_seeks_7d_sum",
        "total_seeks_14d_sum",
        "total_seeks_28d_sum",
        "total_seeks_7d_mean",
        "total_seeks_28d_mean",
        "total_seeks_7d_ewma",
        "total_seeks_14d_ewma",
        "total_seeks_28d_ewma",
        "total_seeks_7d_std",
        "total_seeks_14d_std",
        "total_seeks_28d_std",
        "total_seeks_7d_asfd",
        "total_seeks_14d_asfd",
        "total_seeks_7d_max",
        "total_seeks_14d_max",
        "total_seeks_7d_zscore",
        "total_seeks_14d_zscore",
        "total_seeks_28d_zscore"
    ],
    "총 읽기량": [
        "s242_14d_asfd",
        "s242_14d_ewma",
        "s242_14d_max",
        "s242_14d_std",
        "s242_14d_sum",
        "s242_14d_zscore",
        "s242_28d_accel",
        "s242_28d_ewma",
        "s242_28d_max",
        "s242_28d_mean",
        "s242_28d_std",
        "s242_28d_sum",
        "s242_28d_zscore",
        "s242_7d_accel",
        "s242_7d_asfd",
        "s242_7d_ewma",
        "s242_7d_max",
        "s242_7d_mean",
        "s242_7d_std",
        "s242_7d_sum",
        "total_reads",
        "total_reads_14d_std",
        "total_reads_14d_zscore",
        "total_reads_28d_accel",
        "total_reads_28d_ewma",
        "total_reads_28d_sum",
        "total_reads_28d_zscore",
        "total_reads_7d_accel",
        "total_reads_7d_ewma",
        "total_reads_7d_mean",
        "total_reads_7d_sum",
        "total_reads_7d_zscore"
    ],
    "총 기록량": [
        "s241_7d_accel",
        "s241_28d_accel",
        "s241_7d_sum",
        "s241_14d_sum",
        "s241_28d_sum",
        "s241_7d_mean",
        "s241_28d_mean",
        "s241_7d_ewma",
        "s241_14d_ewma",
        "s241_28d_ewma",
        "s241_7d_std",
        "s241_14d_std",
        "s241_28d_std",
        "s241_14d_max",
        "s241_28d_max",
        "s241_7d_asfd",
        "s241_14d_asfd",
        "s241_14d_zscore",
        "s241_28d_zscore"
    ],
    "Reallocated / Pending": [
        "s5_14d_max",
        "s5_14d_sum",
        "s5_28d_sum"
    ],
    "Sector 열화": [
        "s198_error_rate",
        "s187_error_rate",
        "s197_28d_sum",
        "s198_28d_sum",
        "s197_28d_max",
        "s197_14d_sum",
        "s198_14d_sum",
        "s197_14d_max",
        "s198_14d_max",
        "s197_diff"
    ],
    "읽기/쓰기 안정성": [
        "s183_14d_sum",
        "s183_28d_sum"
    ],
    "Seek 경로 이상": [
        "seek_error_count_diff",
        "seek_error_14d_spike_ratio"
    ],
    "기본 I/O 이상": [
        "smart_199_raw",
        "timeout_total",
        "timeout_total_14d_sum",
        "timeout_total_diff",
        "timeout_total_28d_sum"
    ],
    "열 스트레스": [
        "s190_7d_asfd",
        "s190_7d_cid",
        "s190_14d_asfd",
        "s190_14d_cid",
        "s190_28d_asfd",
        "s190_28d_cid",
        "smart_190_raw"
    ],
    "온도 수준": [
        "s194_7d_asfd",
        "s194_7d_cid",
        "s194_14d_asfd",
        "s194_14d_cid",
        "s194_28d_asfd",
        "s194_28d_cid"
    ],
    "시스템성 실패": [
        "s184_3d_sum",
        "s184_7d_sum",
        "s184_14d_sum",
        "s184_diff"
    ],
    "직접 손상 발생": [
        "smart_5_raw",
        "s5_daily_failure_speed"
    ]
}

# 제외할 피처 목록 병합
exclude_set = set()
for category, feat_list in EXCLUSION_CONFIG.items():
    exclude_set.update(feat_list)
drop_cols = list(exclude_set)

print(f"Info: total drop features count: {len(drop_cols)}")

# ---------------------------------------------
# 3. 데이터 로드 및 피처 필터링
# ---------------------------------------------
print("loading dataset...")
train_df = duckdb.query(f"SELECT * FROM '{train_path}' WHERE date >= '2014-04-01'").df()
val_df  = duckdb.query(f"SELECT * FROM '{val_path}' WHERE date >= '2014-04-01'").df()

print(f"  - Train: {train_df.shape[0]:,} rows, {train_df.shape[1]} cols")
print(f"  - Validation:  {val_df.shape[0]:,} rows, {val_df.shape[1]} cols")

print("Applying filtering...")
drop_cols_lower = {x.lower() for x in drop_cols}
actual_drop_cols = [c for c in train_df.columns if c.lower() in drop_cols_lower]
print(f"  - Matched and dropping {len(actual_drop_cols)} actual columns.")
train_df = train_df.drop(columns=actual_drop_cols, errors='ignore')
val_df  = val_df.drop(columns=actual_drop_cols, errors='ignore')
print(f"  - Filtered Train: {train_df.shape[0]:,} rows, {train_df.shape[1]} cols")
print(f"  - Filtered Validation:  {val_df.shape[0]:,} rows, {val_df.shape[1]} cols")

# ---------------------------------------------
# 4. 필터링된 데이터 저장
# ---------------------------------------------
print("Saving to parquet...")
train_df.to_parquet(TRAIN_OUT, index=False)
val_df.to_parquet(VAL_OUT, index=False)

print("Success!")

Info: total drop features count: 139
loading dataset...
  - Train: 5,141,665 rows, 310 cols
  - Validation:  1,247,377 rows, 310 cols
Applying filtering...
  - Matched and dropping 139 actual columns.
  - Filtered Train: 5,141,665 rows, 171 cols
  - Filtered Validation:  1,247,377 rows, 171 cols
Saving to parquet...
Success!


## 3. 최종 피처 필터링 무결성 검증 테스트 (Verification Validations)

필터링을 마친 출력물인 `fs_train.parquet` 및 `fs_validation.parquet`의 스키마, 데이터 범위, 결측치, 컷오프 일자 및 제외 목록 반영 여부를 통합적으로 철저히 검증합니다.

In [2]:
import duckdb
import os

con = duckdb.connect()
TRAIN_OUT = "../data2/05_feature_selection/fs_train_filtered.parquet"
VAL_OUT  = "../data2/05_feature_selection/fs_validation_filtered.parquet"

print("Verification started...")

try:
    # 1. 파일 물리 존재 검증
    print("Validation 1: Check output files existence")
    assert os.path.exists(TRAIN_OUT), f"Error: {TRAIN_OUT} not found."
    assert os.path.exists(VAL_OUT), f"Error: {VAL_OUT} not found."
    print("  -> [PASS] Output files exist.")

    # 2. 날짜 컷오프 조건 검증
    print("Validation 2: Check date cutoff (>= 2014-04-01)")
    for name, path in [("TRAIN", TRAIN_OUT), ("VAL", VAL_OUT)]:
        min_date = con.execute(f"SELECT MIN(date) FROM read_parquet('{path}')").fetchone()[0]
        min_date_str = min_date.strftime('%Y-%m-%d') if hasattr(min_date, 'strftime') else str(min_date)
        assert min_date_str >= '2014-04-01', f"Error: {name} contains date before cutoff: {min_date_str}"
    print("  -> [PASS] Date cutoff verified.")

    # 3. 제외 피처 삭제 검증
    print("Validation 3: Check excluded features removal")
    exclude_set = set()
    for feat_list in EXCLUSION_CONFIG.values():
        exclude_set.update(feat_list)
        
    for name, path in [("TRAIN", TRAIN_OUT), ("VAL", VAL_OUT)]:
        cols = [c.lower() for c in con.execute(f"SELECT * FROM read_parquet('{path}') WHERE 1=0").df().columns.tolist()]
        exclude_set_lower = {x.lower() for x in exclude_set}
        intersection = exclude_set_lower.intersection(cols)
        assert len(intersection) == 0, f"Error: {name} contains features that should be dropped: {intersection}"
    print("  -> [PASS] All drop features removed successfully.")

    # 4. 데이터셋 비어있지 않음 검증
    print("Validation 4: Check row count > 0")
    for name, path in [("TRAIN", TRAIN_OUT), ("VAL", VAL_OUT)]:
        cnt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
        assert cnt > 0, f"Error: {name} is empty."
        print(f"  -> {name}: {cnt:,} rows")
    print("  -> [PASS] Output datasets are not empty.")

    # 5. Train/Validation 스키마 일치 검증
    print("Validation 5: Check schema match between Train and Validation")
    train_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{TRAIN_OUT}')").fetchdf()
    validation_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{VAL_OUT}')").fetchdf()
    assert train_schema['column_name'].tolist() == validation_schema['column_name'].tolist(), "Error: column names mismatch!"
    assert train_schema['column_type'].tolist() == validation_schema['column_type'].tolist(), "Error: column types mismatch!"
    print("  -> [PASS] Train and Validation schemas are identical.")

    # 6. 결측치(NaN) 및 무한대(Inf) 잔존 여부 정밀 검증
    print("Validation 6: Check NaN/Inf values")
    for name, path in [("TRAIN", TRAIN_OUT), ("VAL", VAL_OUT)]:
        all_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchdf()
        numeric_cols = [r['column_name'] for _, r in all_cols.iterrows() if r['column_name'] not in ('serial_number', 'date') and 'VARCHAR' not in r['column_type']]
        nan_checks = [f"SUM(CASE WHEN isnan({nc}) OR isinf({nc}) THEN 1 ELSE 0 END)" for nc in numeric_cols]
        if nan_checks:
            nan_results = con.execute(f"SELECT { ', '.join(nan_checks) } FROM read_parquet('{path}')").fetchone()
            bad_cols = [numeric_cols[j] for j, v in enumerate(nan_results) if v and v > 0]
            assert len(bad_cols) == 0, f"Error: {name} contains NaN/Inf in columns: {bad_cols}"
    print("  -> [PASS] No NaN/Inf values found in numeric features.")

    # 7. failure 컬럼 값 범위 검증
    print("Validation 7: Check binary values in failure column")
    for name, path in [("TRAIN", TRAIN_OUT), ("VAL", VAL_OUT)]:
        fv = con.execute(f"SELECT DISTINCT failure FROM read_parquet('{path}')").fetchall()
        fset = set(r[0] for r in fv)
        assert fset.issubset({0, 1}), f"Error: {name} failure column contains invalid values: {fset}"
    print("  -> [PASS] failure column is binary (0/1).")

    print("\nSUCCESS: All verification validations passed! (7/7 PASS)")
finally:
    con.close()


Verification started...
Validation 1: Check output files existence
  -> [PASS] Output files exist.
Validation 2: Check date cutoff (>= 2014-04-01)
  -> [PASS] Date cutoff verified.
Validation 3: Check excluded features removal
  -> [PASS] All drop features removed successfully.
Validation 4: Check row count > 0
  -> TRAIN: 5,141,665 rows
  -> VAL: 1,247,377 rows
  -> [PASS] Output datasets are not empty.
Validation 5: Check schema match between Train and Validation
  -> [PASS] Train and Validation schemas are identical.
Validation 6: Check NaN/Inf values
  -> [PASS] No NaN/Inf values found in numeric features.
Validation 7: Check binary values in failure column
  -> [PASS] failure column is binary (0/1).

SUCCESS: All verification validations passed! (7/7 PASS)
